# 0 — Leakage Analysis (Sentetik Attitude × `Converted`)

> **Yazılı karşılığı:** [`docs/0_synthetic_data_and_leakage.docx`](../docs/0_synthetic_data_and_leakage.docx) — tüm metodoloji, gerekçeler ve sayısal sonuçların açıklamalı tartışması orada. Bu notebook yalnız sayıları + grafikleri canlı koşturur.

Çalışan kod, `scripts/leakage_analysis.py` ile **aynı** `datagen.leakage` modülünü çağırır; iki tarafın sayıları birebir aynıdır.

## Niye bu notebook var?

Sentiment için sentetik notlar üretildi (9240 lead, 4 sınıf). Risk: bu üretim yanlışlıkla sentiment'i `Converted` etiketinin gizli bir kopyası haline getirebilir → modelin train AUC'si patlar, production'da çöker. Aşağıdaki dört soru bu riski **sayısal olarak** kontrol eder:

1. attitude tek başına `Converted`'i ne kadar iyi tahmin ediyor? — AUC çok yüksekse leakage var.
2. attitude × `Converted` korelasyonu gerçekçi bantta mı? — Cramér's V ≈ 0.20–0.40 hedef.
3. Her attitude sınıfında hem dönüşen hem dönüşmeyen lead var mı? — yoksa sınıf, label'ı deterministik belirliyor demektir.
4. Ham veri tarafında sonradan dolan / outcome sızdıran kolonlar (`Tags`, `Lead Quality`, `Last Notable Activity`) ne kadar tehlikeli? — lead-scoring modelinde drop edilmeleri gerek mi?

In [ ]:
import sys
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from lead_priority.settings import REPO_ROOT
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
from datagen.leakage import compute_leakage_report

INTERACTIONS = REPO_ROOT / 'data' / 'synthetic' / 'interactions.parquet'
RAW = REPO_ROOT / 'data' / 'Lead Scoring.csv'

interactions = pd.read_parquet(INTERACTIONS)
interactions = interactions[interactions['text'].str.len() > 0].reset_index(drop=True)
raw = pd.read_csv(RAW)
print(f'interactions: {len(interactions)} rows')
print(f'raw         : {len(raw)} rows')
interactions.head()

In [ ]:
report = compute_leakage_report(interactions, raw)
print(f'attitude→Converted AUC (5-fold CV): {report.attitude_to_converted_auc:.3f}')
print(f"Cramér's V (attitude, Converted)   : {report.cramers_v:.3f}")
print(f'Mutual information (nats)          : {report.mutual_information_nats:.4f}')

## 1. Attitude dağılımı

İlk kontrol: 4 sınıf da yeterli temsil ediliyor mu? Tek bir sınıf domine ederse classifier diğerlerini öğrenmekte zorlanır. Hedef: hiçbir sınıf %5'ten az olmasın.

In [ ]:
dist = pd.Series(report.attitude_distribution).sort_values(ascending=False)
ax = dist.plot.bar(color='#4c72b0', figsize=(7, 3.5))
ax.set_ylabel('lead count')
ax.set_title('Attitude class distribution')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()
dist

## 2. Crosstab: attitude × Converted

Her attitude sınıfı içinde **hem dönüşen hem dönüşmeyen** lead bulunmalı. Eğer bir sınıfta sadece converted=1 ya da sadece converted=0 varsa, sentiment etiketi `Converted`'in deterministik fonksiyonu demektir — yani leakage.

Sağlıklı sonuç şuna benzemeli: pozitif sınıfta dönüşüm oranı yüksek ama %100 değil; disengaged'da düşük ama %0 değil. Aralarındaki fark "gerçekçi korelasyon" olmalı, "deterministik kopyalama" değil.

In [ ]:
crosstab_df = pd.DataFrame(report.crosstab).set_index('attitude')
crosstab_df

In [ ]:
pct_cols = [c for c in crosstab_df.columns if c.endswith('_pct')]
ax = crosstab_df[pct_cols].plot.bar(stacked=True, figsize=(7, 3.8), color=['#dd8452', '#4c72b0'])
ax.set_ylabel('% within attitude')
ax.set_title('Conversion rate within each attitude class')
ax.legend(['converted=0', 'converted=1'], loc='upper right')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()

## 3. Tabular leakage adayları (asıl tehlike)

Sentetik sentiment verisi tarafında her şey temiz olsa bile, **ham X Education tabular setinde** lead-scoring modelini zehirleyebilecek kolonlar var. Tipik örnek: `Tags = "Closed by Horizzon"` — bu değer ancak lead'in akıbeti netleştikten sonra giriliyor. Modele bu kolonu verirsen "tahmin etmek" yerine "okumak" yapar; train AUC patlar, production'da çöker çünkü yeni lead için bu kolon boş olur.

Aşağıdaki test basit: tek bir kolonu one-hot encode edip Logistic Regression'a ver, AUC'yi ölç. **0.85 üstü ⇒ kuvvetli leakage sinyali** (siyah kesik çizgi). Lead-scoring modelinde drop edilmesi gereken kolonlar buradan çıkar.

In [ ]:
suspects = pd.DataFrame(report.tabular_leakage_suspects)
suspects

In [ ]:
ax = suspects.set_index('column')['auc'].plot.barh(color='#c44e52', figsize=(7, 3))
ax.axvline(0.85, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('single-column AUC')
ax.set_title('Outcome-leaking column suspects (dashed line: 0.85 alarm)')
plt.tight_layout()
plt.show()

## 4. Otomatik yorum

`compute_leakage_report` yukarıdaki sayılara bakıp insan-okunur bayraklar üretir: AUC sağlıklı bantta mı, Cramér's V hedefte mi, sınıf-coverage boşluğu var mı, hangi tabular kolonlar drop edilmeli. Karar otomatize edilmez — sadece dikkat çekilir.

In [ ]:
for note in report.interpretation:
    print('•', note)